In [24]:
%pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 14.6 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Importing the necessary libraries
import os  # For working with files and directories
import torch  # PyTorch
import torch.nn as nn  # Neural network module
import torchvision.transforms as transforms  # image transformation
from torchvision.models import resnet50, ResNet50_Weights  # Importing the ResNet50 model
from PIL import Image  # image processing library
import psycopg2  # PostgreSQL database adapter

In [ ]:
def load_model():
    """
    Loads the pre-trained ResNet50 model and sets it to evaluation mode.

    Returns:
        model: The loaded ResNet50 model.
    """
    # Use the recommended weights argument instead of pretrained
    weights = ResNet50_Weights.IMAGENET1K_V1  # Alternatively, use ResNet50_Weights.DEFAULT for the latest weights
    model = resnet50(weights=weights)  # Load the model with specified weights
    
    # Remove the last fully connected layer to get 2048-dimensional outputs
    model = torch.nn.Sequential(*list(model.children())[:-1])  # Keep all layers except the last one
    model.eval()  # Set the model to evaluation mode
    return model

In [48]:
def preprocess_image(image_path):
    """
    Preprocess the image.

    parameters:
        image_path (str): Image File Path

    Returns:
        torch.Tensor: Preprocessed image tensor
    """
    # Defining Image Transformation Operations
    transform = transforms.Compose([
        transforms.Resize(256),  # Resizing images
        transforms.CenterCrop(224),  # Center Cropped Image
        transforms.ToTensor(),  # Converting images to tensors
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # standardization
    ])
    
    image = Image.open(image_path).convert("RGB")  # Read images and convert to RGB format
    image_tensor = transform(image).unsqueeze(0)  # Add Batch Dimension
    return image_tensor

In [49]:
def generate_image_vector(model, image_path):
    """
    Generate a 2048-dimensional vector of images.

    parameters:
        model: Pre-trained ResNet50 model
        image_path (str): Image File Path

    Returns:
        numpy.ndarray: 2048-dimensional image vector
    """
    image_tensor = preprocess_image(image_path)  # Preprocessed images
    with torch.no_grad():  # Disable gradient calculation
        vector = model(image_tensor).numpy()  # Generate vectors using models and convert to numpy arrays
        flattened_vector= vector.flatten()
    return flattened_vector # Flatten vectors for easy storage


In [50]:
def search_similar_images(connection, input_vector, top_k):
    """
    Search for the most similar images in the database based on the input vector.

    Args:
        connection: The database connection.
        input_vector: The 2048-dimensional vector of the input image.
        top_k (int): The number of most similar images to retrieve.

    Returns:
        list: A list of tuples containing the article_id, product_code, vector, and distance.
    """
    vector_str = ','.join(map(str, input_vector))  # Convert vector to string format for query
    vector_str = f'[{vector_str}]'
    
    query = """
    SELECT i.image_name, i.vector::text, i.vector <=> %s AS distance
    FROM image_info i
    ORDER BY distance
    LIMIT %s;
    """
    
    with connection.cursor() as cursor:
        cursor.execute(query, (vector_str, top_k))
        results = cursor.fetchall()
    
    return results


In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import ast

def visualize_vectors_interactive(user_vector, similar_results):
    """
    Interactive 3D visualization of the user vector and top_k similar vectors using t-SNE and Plotly.

    Args:
        user_vector (list): The 2048-dimensional vector of the input image.
        similar_results (list): List of tuples containing image_name, vector, and distance.
        
    Returns:
        None
    """
    max_perplexity=50 # max_perplexity (int): The maximum value for perplexity, default is 50.

    print("user_image_vector: ", user_vector)
    print("similar_results_fromDB: ", similar_results)

    # Convert vectors from the results
    extract_similar_vectors = np.array([np.array(ast.literal_eval(result[1])) for result in similar_results])
    print("extract_similar_vectors: ", extract_similar_vectors)
    
    # Combine the user's vector with the similar vectors
    all_vectors = np.vstack([user_vector, extract_similar_vectors])

    # Adjust perplexity based on the number of vectors
    perplexity_value = min(max_perplexity, len(all_vectors) // 2)

    # Use t-SNE to reduce dimensionality to 3D
    tsne = TSNE(n_components=3, perplexity=perplexity_value, random_state=42)
    reduced_vectors = tsne.fit_transform(all_vectors)

    # Create an interactive 3D plot using Plotly
    fig = go.Figure()

    # User vector in red
    fig.add_trace(go.Scatter3d(
        x=[reduced_vectors[0, 0]], y=[reduced_vectors[0, 1]], z=[reduced_vectors[0, 2]],
        mode='markers+text',
        marker=dict(size=10, color='red'),
        text="User Vector",
        textposition="top center"
    ))

    # Similar vectors in blue
    fig.add_trace(go.Scatter3d(
        x=reduced_vectors[1:, 0], y=reduced_vectors[1:, 1], z=reduced_vectors[1:, 2],
        mode='markers+text',
        marker=dict(size=5, color='blue'),
        text=[result[0] for result in similar_results],  # image_name as labels
        textposition="top center"
    ))

    fig.update_layout(
        title="Interactive 3D t-SNE Visualization of User Vector and Similar Vectors",
        scene=dict(
            xaxis_title="t-SNE Dimension 1",
            yaxis_title="t-SNE Dimension 2",
            zaxis_title="t-SNE Dimension 3"
        ),
        width=1200,  # Set the width of the plot
        height=1000  # Set the height of the plot
    )
    fig.show()


In [46]:
import psycopg2  # PostgreSQL database adapter

# Main search and visualize function

TOP_K = 100  # Number of similar images to retrieve

image_path = "/Users/Tommy/AI/image-search/clothing-images/050/0501619010.jpg"
# image_path = "/Users/Tommy/Downloads/man-pants.png"
# image_path = "/Users/Tommy/Snipaste/Snipaste_2024-10-24_12-54-18.png"
# image_path = "/Users/Tommy/AI/image-search/clothing-images/050/0501619010.jpg"

# Connecting to PostgreSQL Database
connection = psycopg2.connect(
        dbname='image-search',
        user='postgres',
        password='test-postgres',
        host='localhost',
        port='5432'
    )

model = load_model()  # Load the ResNet50 model

# Step 1: Extract user vector from the image
flattened_user_vector = generate_image_vector(model, image_path) 
# flattened_vector:  [0.3435464  0.73842794 0.91404784 ... 0.02022145 0.50024617 0.42373607]

# Step 2: Find similar images
similar_images_vector = search_similar_images(connection, flattened_user_vector, TOP_K)

# Step 3: Visualize the user vector and similar images
# visualize_vectors(flattened_user_vector, similar_images_vector)

visualize_vectors_interactive(flattened_user_vector, similar_images_vector)

connection.close()  # Close the database connection


user_image_vector:  [0.3435464  0.73842794 0.91404784 ... 0.02022145 0.50024617 0.42373607]
similar_results_fromDB:  [('0501619010', '[0.3435464,0.73842794,0.91404784,0.057762798,0.29810685,0.053511746,2.6018279,0.28095376,0.45440212,0.8106928,0.49308196,0.045211278,0.095176995,0.4936936,0.93651605,0.024817966,0.13418137,0.016092636,0.16448504,0.18342866,0.15826853,0.7044571,0.39026615,0.008719066,0.17506845,0.22308782,0.48572576,0.21259974,0.15775508,0.09245795,0.73023623,0.015314922,0.12398153,0.17651324,0.20156996,0.021751214,0.095195234,2.71005,0.1662065,0.30179656,0.31303617,0.32400775,0.37269327,0.06938759,0.33953956,0.02776832,0.1744089,1.9534751,0.13229115,0.24477825,0.3602861,0.1723407,0.37732026,0.09229229,0.3917375,0.15555184,0.028219247,0.020092592,0.11957594,0.4592939,0.06585371,0.06484489,0.4100548,0.0019747443,0.07178196,0.4669806,0.11185668,0.5276334,0.14954488,0.6605945,0.92311877,0.0046075885,0.032105803,0.58987576,0.41393036,0.13785315,0.7116949,0.7756844,0.13741817,